### Import libraries

In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    Dense,
    GlobalAveragePooling2D,
    Dropout
)
import sys
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from pathlib import Path
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import create_dataset, IMG_SIZE, BATCH_SIZE

print("TensorFlow version:", tf.__version__)
print("Project root:", PROJECT_ROOT)


TensorFlow version: 2.21.0
Project root: d:\Coding Files\PlantDiseaseDetection


### Define paths and classes

In [3]:
DATA_PATH = PROJECT_ROOT / "dataset" / "processed"
MODEL_PATH = PROJECT_ROOT / "models"
GRAPH_PATH = PROJECT_ROOT / "results" / "graphs"

MODEL_PATH.mkdir(parents=True, exist_ok=True)
GRAPH_PATH.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_healthy"
]

NUM_CLASSES = len(CLASSES)

print("Number of classes:", NUM_CLASSES)

Number of classes: 5


### Load CSV files

In [4]:
train_df = pd.read_csv(DATA_PATH / "train.csv")
validation_df = pd.read_csv(DATA_PATH / "validation.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")

print("Training images:", len(train_df))
print("Validation images:", len(validation_df))
print("Testing images:", len(test_df))

Training images: 5305
Validation images: 1137
Testing images: 1137


### Create TensorFlow datasets

In [5]:
train_dataset = create_dataset(
    train_df,
    CLASSES,
    training=True
)

validation_dataset = create_dataset(
    validation_df,
    CLASSES,
    training=False
)

test_dataset = create_dataset(
    test_df,
    CLASSES,
    training=False
)
print("Training dataset:", train_dataset)
print("Validation dataset:", validation_dataset)
print("Testing dataset:", test_dataset)

Training dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
Validation dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
Testing dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


### Load pretrained MobileNetV2

In [6]:
IMG_SIZE = (224, 224)
NUM_CLASSES = 5

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

### Freeze pretrained layers

In [7]:
base_model.trainable = False

### Build the classification model

In [8]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

x = Dropout(0.3)(x)

predictions = Dense(
    NUM_CLASSES,
    activation='softmax'
)(x)

model_mobilenet = Model(
    inputs=base_model.input,
    outputs=predictions
)

### Compile the model

In [9]:
model_mobilenet.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_mobilenet.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [19]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_dataset_mobilenet = train_dataset.map(
    lambda x, y: (preprocess_input(x * 255.0), y)
)

validation_dataset_mobilenet = validation_dataset.map(
    lambda x, y: (preprocess_input(x * 255.0), y)
)

In [20]:
for images, labels in train_dataset_mobilenet.take(1):

    print("Minimum:", tf.reduce_min(images).numpy())
    print("Maximum:", tf.reduce_max(images).numpy())

Minimum: -1.0
Maximum: 1.0312865


### Train the model

In [21]:
EPOCHS = 10

history_mobilenet = model_mobilenet.fit(
    train_dataset_mobilenet,
    validation_data=validation_dataset_mobilenet,
    epochs=EPOCHS
)

Epoch 1/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 90s 267ms/step - accuracy: 0.6794 - loss: 0.9025 - val_accuracy: 0.6966 - val_loss: 0.8464
Epoch 2/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 92s 274ms/step - accuracy: 0.8398 - loss: 0.4898 - val_accuracy: 0.8276 - val_loss: 0.4989
Epoch 3/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 89s 265ms/step - accuracy: 0.8779 - loss: 0.3750 - val_accuracy: 0.8874 - val_loss: 0.3320
Epoch 4/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 83s 248ms/step - accuracy: 0.8992 - loss: 0.3057 - val_accuracy: 0.9094 - val_loss: 0.2512
Epoch 5/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 87s 258ms/step - accuracy: 0.9161 - loss: 0.2608 - val_accuracy: 0.9173 - val_loss: 0.2200
Epoch 6/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 98s 292ms/step - accuracy: 0.9242 - loss: 0.2342 - val_accuracy: 0.9252 - val_loss: 0.1996
Epoch 7/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 93s 275ms/step - accuracy: 0.9318 - loss: 0.2078 - val_accuracy: 0.9376 - val_loss: 0.1830
Epoch 8/10
332/332 ━━━━━━━━━━━━━━━━━━━━ 93s 277ms/step - accuracy: 0.9393 - loss: 0

### Fine-tuning

###### After initial training, we'll unfreeze some of the deeper MobileNetV2 layers to adapt them to plant disease images.

In [22]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

###### Recompile with a smaller learning rate:

In [23]:
model_mobilenet.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

###### Train again:

In [24]:
history_finetune = model_mobilenet.fit(
    train_dataset_mobilenet,
    validation_data=validation_dataset_mobilenet,
    epochs=5
)

Epoch 1/5
332/332 ━━━━━━━━━━━━━━━━━━━━ 91s 252ms/step - accuracy: 0.9497 - loss: 0.1571 - val_accuracy: 0.9490 - val_loss: 0.1555
Epoch 2/5
332/332 ━━━━━━━━━━━━━━━━━━━━ 85s 252ms/step - accuracy: 0.9555 - loss: 0.1399 - val_accuracy: 0.9464 - val_loss: 0.1658
Epoch 3/5
332/332 ━━━━━━━━━━━━━━━━━━━━ 85s 254ms/step - accuracy: 0.9553 - loss: 0.1368 - val_accuracy: 0.9472 - val_loss: 0.1645
Epoch 4/5
332/332 ━━━━━━━━━━━━━━━━━━━━ 85s 251ms/step - accuracy: 0.9563 - loss: 0.1349 - val_accuracy: 0.9499 - val_loss: 0.1539
Epoch 5/5
332/332 ━━━━━━━━━━━━━━━━━━━━ 85s 254ms/step - accuracy: 0.9583 - loss: 0.1288 - val_accuracy: 0.9516 - val_loss: 0.1448


### Save the model

In [25]:
model_mobilenet.save(MODEL_PATH / "mobilenetv2.keras")